In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(data_folder: str, **storage_options) -> pd.DataFrame:
    file_path = f"{data_folder}/supplier.tbl"
    return pd.read_csv(
        file_path,
        sep="|",
        engine="c",
        memory_map=True,
        **storage_options
    )

supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

orders_multi = (
    lineitem[['L_ORDERKEY','L_SUPPKEY']]
    .groupby('L_ORDERKEY', sort=False)['L_SUPPKEY']
    .nunique()
)
multi_keys = orders_multi.index[orders_multi>1]

li = lineitem.loc[
    (lineitem['L_ORDERKEY'].isin(multi_keys)) &
    (lineitem['L_RECEIPTDATE'] > lineitem['L_COMMITDATE']),
    ['L_ORDERKEY','L_SUPPKEY']
]

orders_single = li.groupby('L_ORDERKEY', sort=False)['L_SUPPKEY'].nunique()
single_keys = orders_single.index[orders_single==1]

orders_f = orders.loc[orders['O_ORDERSTATUS']=='F','O_ORDERKEY']
saudi_keys = nation.loc[nation['N_NAME']=='SAUDI ARABIA','N_NATIONKEY']

sup_N = supplier.set_index('S_SUPPKEY')['S_NATIONKEY']
sup_name = supplier.set_index('S_SUPPKEY')['S_NAME']

total = (
    li
    .loc[lambda df: df['L_ORDERKEY'].isin(single_keys)]
    .loc[lambda df: df['L_ORDERKEY'].isin(orders_f)]
    [['L_SUPPKEY']]
    .assign(
        S_NATIONKEY=lambda df: df['L_SUPPKEY'].map(sup_N),
        S_NAME=lambda df: df['L_SUPPKEY'].map(sup_name)
    )
    .loc[lambda df: df['S_NATIONKEY'].isin(saudi_keys), ['S_NAME']]
    .groupby('S_NAME', as_index=False, sort=False)
    .size()
    .rename(columns={'size':'NUMWAIT'})
    .sort_values(['NUMWAIT','S_NAME'], ascending=[False,True])
)